In [2]:
pip install huggingface_hub

   ---------------------------------------- 0.0/765.1 kB ? eta -:--:--
   ---------------------------------------- 765.1/765.1 kB 10.7 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 20.1 MB/s  0:00:00

  Attempting uninstall: click

    Found existing installation: click 8.2.1

    Uninstalling click-8.2.1:

      Successfully uninstalled click-8.2.1

   ------------- -------------------------- 1/3 [click]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   -------------------------- ------------- 2/3 [huggingface_hub]
   ----------------------------------

In [12]:
import json
from typing import Any, Dict, List, Optional

import requests


# ============================================================
# 1. CONFIGURATION
# ============================================================

OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"

MODEL = "qwen3:4b"

REQUEST_TIMEOUT_SECONDS = 20
OLLAMA_TIMEOUT_SECONDS = 180
MAX_PREDICTED_TOKENS = 300


# ============================================================
# 2. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are a ReAct-style weather agent.

You have access to one tool named get_weather.

The get_weather tool:
- gets the current weather for a location
- requires one string argument named location

For any current weather question, you must use the tool.
Never answer current weather from memory.


WHEN YOU NEED TO USE THE TOOL

Respond using this format:

Thought:
A brief statement explaining that current weather data is needed.

Action:
A valid JSON object with exactly two top-level keys:
1. "action", whose value must be "get_weather"
2. "action_input", whose value must be an object containing
   one key named "location"

After the JSON object, write exactly:

Observation:

Do not invent the Observation.
The external Python program will execute the tool and provide
the real Observation.


AFTER YOU RECEIVE AN OBSERVATION

If the Observation contains enough weather data to answer
the original question, do not call the tool again.

Return a concise final answer based only on the Observation.

The final answer must begin with exactly:

Final Answer:

Do not include:
- Thought
- Action
- Action JSON
- Observation
- raw JSON
- tool explanations
- process explanations

Do not mention:
- the agent process
- the tool
- Open-Meteo

Use only facts present in the Observation.
""".strip()


# ============================================================
# 3. FINAL ANSWER INSTRUCTION
#
# This is sent with the real Observation before
# the second LLM call.
# ============================================================

FINAL_ANSWER_INSTRUCTION = """
Return only the final user-facing answer.

Your entire response must use exactly this structure:

Final Answer:
<one concise sentence summarizing the weather>

Rules:
- Start immediately with "Final Answer:"
- Do not show reasoning
- Do not say "Okay"
- Do not say "let's see"
- Do not include Thought
- Do not include Action
- Do not include Observation
- Do not include JSON
- Do not explain your process
- Do not discuss the instructions
- Do not mention the tool
- Do not mention Open-Meteo
- Use only facts from the Observation
- Keep the answer under 60 words
""".strip()


# ============================================================
# 4. OUTPUT FORMATTING
# ============================================================

def print_section(
    title: str,
    content: Any,
) -> None:
    """
    Print a consistently formatted output section.
    """

    print()
    print("=" * 70)
    print(title)
    print("=" * 70)
    print(content)


# ============================================================
# 5. CHECK OLLAMA
# ============================================================

def check_ollama() -> None:
    """
    Verify that:
    1. The local Ollama server is reachable.
    2. The required model is installed.
    """

    try:
        response = requests.get(
            OLLAMA_TAGS_URL,
            timeout=10,
        )

        response.raise_for_status()

    except requests.ConnectionError as exc:
        raise RuntimeError(
            "\nCould not connect to Ollama.\n\n"
            "Make sure Ollama is running.\n\n"
            "In PowerShell, try:\n"
            "    ollama serve\n"
        ) from exc

    except requests.RequestException as exc:
        raise RuntimeError(
            f"Ollama health check failed: {exc}"
        ) from exc

    try:
        data = response.json()

    except ValueError as exc:
        raise RuntimeError(
            "Ollama returned invalid JSON "
            "during the health check."
        ) from exc

    installed_models = [
        model.get("name", "")
        for model in data.get("models", [])
    ]

    installed_models_text = (
        "\n".join(
            f"- {model_name}"
            for model_name in installed_models
        )
        or
        "No models found."
    )

    print_section(
        "INSTALLED OLLAMA MODELS",
        installed_models_text,
    )

    if MODEL not in installed_models:
        raise RuntimeError(
            f"\nModel '{MODEL}' is not installed.\n\n"
            "Run:\n"
            f"    ollama pull {MODEL}\n"
        )


# ============================================================
# 6. REAL WEATHER TOOL
#
# Uses Open-Meteo.
# No weather API key required.
# ============================================================

def get_weather(location: str) -> str:
    """
    Get real current weather for a location.

    Flow:
    1. Convert location name to latitude/longitude.
    2. Retrieve current weather using those coordinates.
    3. Return the result as a JSON string.
    """

    # --------------------------------------------------------
    # STEP 1: GEOCODING
    # --------------------------------------------------------

    geocoding_url = (
        "https://geocoding-api.open-meteo.com/v1/search"
    )

    geocoding_params = {
        "name": location,
        "count": 1,
        "language": "en",
        "format": "json",
    }

    try:
        response = requests.get(
            geocoding_url,
            params=geocoding_params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        response.raise_for_status()

    except requests.RequestException as exc:
        return json.dumps(
            {
                "error": (
                    f"Geocoding request failed: {exc}"
                )
            },
            indent=2,
        )

    try:
        geocoding_data = response.json()

    except ValueError:
        return json.dumps(
            {
                "error": (
                    "Geocoding API returned invalid JSON."
                )
            },
            indent=2,
        )

    results = geocoding_data.get("results")

    if not results:
        return json.dumps(
            {
                "error": (
                    f"Could not find location: {location}"
                )
            },
            indent=2,
        )

    place = results[0]

    latitude = place["latitude"]
    longitude = place["longitude"]

    resolved_name = place.get(
        "name",
        location,
    )

    country = place.get(
        "country",
        "",
    )

    # --------------------------------------------------------
    # STEP 2: CURRENT WEATHER
    # --------------------------------------------------------

    weather_url = (
        "https://api.open-meteo.com/v1/forecast"
    )

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "apparent_temperature,"
            "relative_humidity_2m,"
            "precipitation,"
            "wind_speed_10m"
        ),
        "timezone": "auto",
    }

    try:
        response = requests.get(
            weather_url,
            params=weather_params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )

        response.raise_for_status()

    except requests.RequestException as exc:
        return json.dumps(
            {
                "error": (
                    f"Weather request failed: {exc}"
                )
            },
            indent=2,
        )

    try:
        weather_data = response.json()

    except ValueError:
        return json.dumps(
            {
                "error": (
                    "Weather API returned invalid JSON."
                )
            },
            indent=2,
        )

    current = weather_data.get("current")

    if not current:
        return json.dumps(
            {
                "error": (
                    "No current weather data returned."
                )
            },
            indent=2,
        )

    # --------------------------------------------------------
    # STEP 3: STRUCTURED OBSERVATION
    # --------------------------------------------------------

    result = {
        "location": resolved_name,
        "country": country,
        "temperature_c": current.get(
            "temperature_2m"
        ),
        "feels_like_c": current.get(
            "apparent_temperature"
        ),
        "humidity_percent": current.get(
            "relative_humidity_2m"
        ),
        "precipitation_mm": current.get(
            "precipitation"
        ),
        "wind_speed_kmh": current.get(
            "wind_speed_10m"
        ),
    }

    return json.dumps(
        result,
        indent=2,
    )


# ============================================================
# 7. CALL OLLAMA
# ============================================================

def call_llm(
    messages: List[Dict[str, str]],
    stop: Optional[List[str]] = None,
) -> str:
    """
    Call Ollama's local /api/chat endpoint.

    Important behavior:
    - stream=False
    - think=False
    - temperature=0
    - num_predict=300
    - stop sequences are added only when provided
    """

    payload: Dict[str, Any] = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "think": False,
        "options": {
            "temperature": 0,
            "num_predict": MAX_PREDICTED_TOKENS,
        },
    }

    # --------------------------------------------------------
    # Add stop sequences only when explicitly provided
    # --------------------------------------------------------

    if stop is not None:
        payload["options"]["stop"] = stop

    # --------------------------------------------------------
    # Call Ollama
    # --------------------------------------------------------

    try:
        response = requests.post(
            OLLAMA_CHAT_URL,
            json=payload,
            timeout=OLLAMA_TIMEOUT_SECONDS,
        )

        response.raise_for_status()

    except requests.ConnectionError as exc:
        raise RuntimeError(
            "\nCould not connect to Ollama.\n\n"
            "Run:\n"
            "    ollama serve\n"
        ) from exc

    except requests.RequestException as exc:
        raise RuntimeError(
            f"Ollama request failed: {exc}"
        ) from exc

    # --------------------------------------------------------
    # Parse response JSON
    # --------------------------------------------------------

    try:
        data = response.json()

    except ValueError as exc:
        raise RuntimeError(
            "Ollama returned invalid JSON."
        ) from exc

    # --------------------------------------------------------
    # Check Ollama-level error
    # --------------------------------------------------------

    if "error" in data:
        raise RuntimeError(
            f"Ollama error: {data['error']}"
        )

    # --------------------------------------------------------
    # Extract assistant message
    # --------------------------------------------------------

    message = data.get("message")

    if not isinstance(message, dict):
        raise RuntimeError(
            "Ollama response did not contain "
            "a valid message object.\n\n"
            + json.dumps(
                data,
                indent=2,
            )
        )

    content = message.get(
        "content",
        "",
    )

    # --------------------------------------------------------
    # Detect empty output
    # --------------------------------------------------------

    if not content or not content.strip():
        raise RuntimeError(
            "\nOllama returned empty message.content.\n\n"
            "Full response:\n"
            + json.dumps(
                data,
                indent=2,
            )
        )

    return content


# ============================================================
# 8. EXTRACT ACTION JSON
#
# Important:
# We do not return the first arbitrary JSON object.
#
# We only accept a JSON object containing both:
# - "action"
# - "action_input"
# ============================================================

def extract_action_json(
    text: str,
) -> Dict[str, Any]:
    """
    Find the first valid JSON object representing
    an agent action.

    Required keys:
    - action
    - action_input
    """

    decoder = json.JSONDecoder()

    for index, character in enumerate(text):

        if character != "{":
            continue

        try:
            obj, _ = decoder.raw_decode(
                text[index:]
            )

        except json.JSONDecodeError:
            continue

        if not isinstance(obj, dict):
            continue

        if (
            "action" in obj
            and
            "action_input" in obj
        ):
            return obj

    raise ValueError(
        "\nCould not find a valid Action JSON object.\n\n"
        "The model output must contain an object with:\n"
        '- "action"\n'
        '- "action_input"\n\n'
        "Raw model output:\n"
        f"{text!r}"
    )


# ============================================================
# 9. VALIDATE ACTION
# ============================================================

def validate_action(
    action_data: Dict[str, Any],
) -> str:
    """
    Validate the model-generated tool request.

    Returns:
        Cleaned location string.
    """

    if not isinstance(action_data, dict):
        raise ValueError(
            "Action must be a JSON object."
        )

    action_name = action_data.get(
        "action"
    )

    if action_name != "get_weather":
        raise ValueError(
            "Expected action='get_weather', got:\n"
            + json.dumps(
                action_data,
                indent=2,
            )
        )

    action_input = action_data.get(
        "action_input"
    )

    if not isinstance(action_input, dict):
        raise ValueError(
            "'action_input' must be a JSON object."
        )

    location = action_input.get(
        "location"
    )

    if not isinstance(location, str):
        raise ValueError(
            "'location' must be a string."
        )

    location = location.strip()

    if not location:
        raise ValueError(
            "'location' cannot be empty."
        )

    return location


# ============================================================
# 10. EXTRACT FINAL ANSWER
#
# Small local models may produce visible reasoning before
# the actual Final Answer.
#
# We therefore extract only the LAST occurrence of:
#
# Final Answer:
#
# Using rsplit() is intentional because the model may mention
# the phrase earlier while reasoning about the required format.
# ============================================================

def extract_final_answer(
    text: str,
) -> str:
    """
    Extract only the final user-facing answer.

    Everything before the last occurrence of
    'Final Answer:' is discarded.
    """

    marker = "Final Answer:"

    if marker not in text:
        raise ValueError(
            "\nModel output did not contain "
            "'Final Answer:'.\n\n"
            "Raw model output:\n"
            f"{text!r}"
        )

    answer = text.rsplit(
        marker,
        1,
    )[1].strip()

    if not answer:
        raise ValueError(
            "\nModel produced an empty final answer.\n\n"
            "Raw model output:\n"
            f"{text!r}"
        )

    return answer


# ============================================================
# 11. MAIN AGENT FLOW
# ============================================================

def main() -> None:
    """
    Run the complete ReAct-style weather agent flow.

    Flow:
    1. Check Ollama.
    2. Send user question to LLM.
    3. Stop before Observation.
    4. Parse Action JSON.
    5. Validate tool request.
    6. Execute real weather API.
    7. Add assistant action to history.
    8. Add real Observation as a user message.
    9. Ask LLM for final answer.
    10. Extract only the clean Final Answer.
    """

    # --------------------------------------------------------
    # A. CHECK OLLAMA
    # --------------------------------------------------------

    check_ollama()

    # --------------------------------------------------------
    # B. USER QUESTION
    #
    # Same example used in the HF lesson.
    # --------------------------------------------------------

    user_question = (
        "What's the weather in London?"
    )

    # --------------------------------------------------------
    # C. INITIAL CONVERSATION
    # --------------------------------------------------------

    messages: List[Dict[str, str]] = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_question,
        },
    ]

    # --------------------------------------------------------
    # D. FIRST LLM CALL
    #
    # Critical behavior:
    #
    # Stop generation before the model can invent
    # the Observation.
    # --------------------------------------------------------

    first_output = call_llm(
        messages=messages,
        stop=["Observation:"],
    )

    print_section(
        "FIRST LLM OUTPUT",
        first_output,
    )

    # --------------------------------------------------------
    # E. EXTRACT ACTION JSON
    # --------------------------------------------------------

    action_data = extract_action_json(
        first_output
    )

    print_section(
        "PARSED ACTION",
        json.dumps(
            action_data,
            indent=2,
        ),
    )

    # --------------------------------------------------------
    # F. VALIDATE ACTION
    # --------------------------------------------------------

    location = validate_action(
        action_data
    )

    print_section(
        "LOCATION REQUESTED BY MODEL",
        location,
    )

    # --------------------------------------------------------
    # G. EXECUTE REAL WEATHER TOOL
    # --------------------------------------------------------

    observation = get_weather(
        location=location
    )

    print_section(
        "REAL OBSERVATION FROM OPEN-METEO",
        observation,
    )

    # --------------------------------------------------------
    # H. ADD ASSISTANT ACTION TO HISTORY
    #
    # Critical behavior:
    #
    # Keep the model's Thought + Action in its own
    # assistant message.
    # --------------------------------------------------------

    messages.append(
        {
            "role": "assistant",
            "content": first_output,
        }
    )

    # --------------------------------------------------------
    # I. ADD REAL OBSERVATION AS USER MESSAGE
    #
    # Critical behavior:
    #
    # Do NOT merge the Observation into the previous
    # assistant message.
    #
    # Preserve this sequence:
    #
    # system
    # user
    # assistant
    # user
    # assistant
    # --------------------------------------------------------

    messages.append(
        {
            "role": "user",
            "content": (
                "Observation:\n"
                + observation
                + "\n\n"
                + FINAL_ANSWER_INSTRUCTION
            ),
        }
    )

    # --------------------------------------------------------
    # J. DEBUG MESSAGE HISTORY
    # --------------------------------------------------------

    print_section(
        "MESSAGES BEFORE SECOND LLM CALL",
        json.dumps(
            messages,
            indent=2,
        ),
    )

    # --------------------------------------------------------
    # K. SECOND LLM CALL
    #
    # Critical behavior:
    #
    # No stop sequence here.
    #
    # The model must be allowed to generate:
    #
    # Final Answer:
    # ...
    # --------------------------------------------------------

    raw_final_output = call_llm(
        messages=messages,
        stop=None,
    )

    # --------------------------------------------------------
    # L. EXTRACT CLEAN FINAL ANSWER
    #
    # If the model produces reasoning such as:
    #
    # Okay, let's see...
    # ...
    # Final Answer:
    # The current weather...
    #
    # everything before the LAST "Final Answer:"
    # is discarded.
    # --------------------------------------------------------

    final_answer = extract_final_answer(
        raw_final_output
    )

    # --------------------------------------------------------
    # M. PRINT ONLY CLEAN FINAL ANSWER
    # --------------------------------------------------------

    print_section(
        "FINAL ANSWER",
        final_answer,
    )


# ============================================================
# 12. RUN
# ============================================================

if __name__ == "__main__":
    main()


INSTALLED OLLAMA MODELS
- qwen3:4b

FIRST LLM OUTPUT
Okay, the user is asking about the weather in London. I need to use the get_weather tool for this. Let me check the instructions again. The tool requires a location string. So I'll set the action to get_weather with location "London". 

Wait, the tool's name is get_weather, and the action_input should be a JSON object with "location". So the Action JSON should be {"action": "get_weather", "action_input": {"location": "London"}}.

I have to make sure I don't answer from memory. Since the user is asking for current weather, I must use the tool. No other tools are available. So I'll structure the response as per ReAct style.

After getting the observation from the tool, I'll check if the data is sufficient. But for now, the next step is to output the Thought, Action, and then Observation. Wait, the user's message is the query, so I need to generate the Thought first.

Thought: A brief statement that current weather data is needed for L